In [34]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', None)  # show all columns
pd.set_option('display.max_rows', 100)      # show up to 100 rows

In [35]:
import os
print(os.getcwd())
print(os.listdir())

c:\Users\User\Desktop\CreditRiskLearning\Exploration
['exploration.ipynb', 'modelling.ipynb']


In [36]:
# Build path relative to notebook location
base_path = os.path.dirname(os.path.abspath('exploration.ipynb'))
data_path = os.path.join(base_path, 'German Credit Data', 'german.data')


In [37]:
columns = [
    'checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount',
    'savings', 'employment', 'installment_rate', 'personal_status', 'other_debtors',
    'residence_since', 'property', 'age', 'other_installments', 'housing',
    'existing_credits', 'job', 'dependents', 'telephone', 'foreign_worker', 'target'
]



In [38]:
# Load data
df = pd.read_csv('../German Credit Data/german.data', sep=' ', header=None, names=columns)

# Map target FIRST before anything else
df['target'] = df['target'].map({1: 0, 2: 1})

# Verify it worked
print(df['target'].unique())  # should show [0 1]

# Then encode
cat_columns = df.select_dtypes(include='object').columns.tolist()
df_encoded = pd.get_dummies(df, columns=cat_columns, drop_first=True)

# Then split
X = df_encoded.drop('target', axis=1)
y = df_encoded['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Verify split target values
print(y_test.unique())  # should show [0 1]

[0 1]
[1 0]


C:\Users\User\AppData\Local\Temp\ipykernel_16784\195248330.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_columns = df.select_dtypes(include='object').columns.tolist()


In [39]:
df.dtypes

checking_status         str
duration              int64
credit_history          str
purpose                 str
credit_amount         int64
savings                 str
employment              str
installment_rate      int64
personal_status         str
other_debtors           str
residence_since       int64
property                str
age                   int64
other_installments      str
housing                 str
existing_credits      int64
job                     str
dependents            int64
telephone               str
foreign_worker          str
target                int64
dtype: object

In [40]:
# Get list of categorical items
cat_columns = df.select_dtypes(include='object').columns.tolist()
print(cat_columns)

# One hoit encode items
df_encoded = pd.get_dummies(df, columns=cat_columns, drop_first=True)

print(f'Original shape: {df.shape}')
print(f'Encoded shape: {df_encoded.shape}')

['checking_status', 'credit_history', 'purpose', 'savings', 'employment', 'personal_status', 'other_debtors', 'property', 'other_installments', 'housing', 'job', 'telephone', 'foreign_worker']
Original shape: (1000, 21)
Encoded shape: (1000, 49)


C:\Users\User\AppData\Local\Temp\ipykernel_16784\3149441456.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_columns = df.select_dtypes(include='object').columns.tolist()


In [41]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Split into features and target
X = df_encoded.drop('target', axis=1)
y = df_encoded['target']


# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training rows: {X_train.shape[0]}')
print(f'Testing rows: {X_test.shape[0]}')

Training rows: 800
Testing rows: 200


In [42]:
# Build and train the model

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

# Make predictions

y_pred = lr_model.predict(X_test)

print('Model trained successfully!')


Model trained successfully!


c:\Users\User\Desktop\CreditRiskLearning\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [43]:
from sklearn.preprocessing import StandardScaler

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Retrain with scaled data and more iterations

lr_model = LogisticRegression(max_iter = 2000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

y_pred = lr_model.predict(X_test_scaled)

print('Model trained successfully!')

Model trained successfully!


In [44]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.84      0.88      0.86       141
           1       0.67      0.59      0.63        59

    accuracy                           0.80       200
   macro avg       0.76      0.74      0.74       200
weighted avg       0.79      0.80      0.79       200



In [ ]:
# Get probabilities instead of hard predictions

y_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

# Try a lower threshold

threshold = 0.3
y_pred_tuned = (y_prob >= threshold).astype(int)

print(f'Results at threshold {threshold}:')
print(classification_report(y_test, y_pred_tuned))


Results at threshold 0.4:
              precision    recall  f1-score   support

           0       0.87      0.82      0.85       141
           1       0.63      0.71      0.67        59

    accuracy                           0.79       200
   macro avg       0.75      0.77      0.76       200
weighted avg       0.80      0.79      0.79       200

